# 01 - Descarga de datos

TFM: analisis del descenso del RCD Mallorca en LaLiga 2025/26.

Este notebook descarga y guarda en disco todos los datos brutos. **Se ejecuta una sola vez.**
Los ficheros ya descargados se detectan y se saltan.

| Fuente | Contenido | Navegador |
|---|---|---|
| Understat | Equipos, jugadores, disparos con xG | No |
| FBref | Estadisticas de jugador (Primera y Segunda) | Si (Selenium) |
| WhoScored | Eventos Opta de los 380 partidos de LaLiga | Si (Selenium) |

**Limitacion documentada:** WhoScored no publica eventos para Segunda Division
(las fichas devuelven contenido vacio), por lo que esa categoria solo se cubre con FBref.

## 1. Instalacion de librerias

In [ ]:
# Instalacion de las librerias necesarias (ejecutar solo la primera vez)
get_ipython().system('pip install soccerdata pandas matplotlib seaborn mplsoccer pyarrow openpyxl scikit-learn')

## 2. Importaciones y estructura de carpetas

In [ ]:
# Subimos a la carpeta raiz del proyecto para que las rutas relativas funcionen
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Importamos la libreria de scraping deportivo
import soccerdata as sd
# Importamos pandas para manejar los datos
import pandas as pd
# Importamos os para gestionar carpetas y rutas
import os
# Importamos glob para listar ficheros
import glob
# Importamos json para leer y escribir configuracion
import json
# Importamos logging para reducir los mensajes informativos
import logging

# Silenciamos los mensajes informativos de soccerdata
logging.getLogger("soccerdata").setLevel(logging.WARNING)

# Creamos la estructura de carpetas del proyecto
for carpeta in ["data", "data/eventos", "data/export_bi", "notebooks", "images"]:
    os.makedirs(carpeta, exist_ok=True)

# Mostramos la carpeta de trabajo para verificar la ubicacion
print("Carpeta de trabajo:", os.getcwd())

## 3. Understat: equipos, jugadores y disparos

Fuente sin navegador, la mas rapida. Cubre LaLiga pero no Segunda Division.

In [ ]:
# Creamos el lector de Understat para LaLiga temporada 2025-2026
understat = sd.Understat(leagues="ESP-La Liga", seasons="2025-2026")

# Descargamos el calendario completo de la temporada
calendario = understat.read_schedule()
# Guardamos el calendario
calendario.to_csv("data/calendario_laliga_2526.csv")

# Descargamos las estadisticas de equipo partido a partido
equipos = understat.read_team_match_stats()
# Guardamos las estadisticas de equipo
equipos.to_csv("data/equipos_partido_laliga_2526.csv")

# Descargamos las estadisticas de jugador de toda la temporada
jugadores = understat.read_player_season_stats()
# Guardamos las estadisticas de jugador
jugadores.to_csv("data/jugadores_laliga_2526.csv")

# Comprobamos las dimensiones de lo descargado
print("Calendario:", calendario.shape)
print("Equipos por partido:", equipos.shape)
print("Jugadores:", jugadores.shape)

In [ ]:
# Descargamos todos los disparos con coordenadas y xG de la temporada
disparos = understat.read_shot_events()

# Guardamos los disparos en bruto
disparos.to_csv("data/disparos_laliga_2526.csv")

# Comprobamos las dimensiones
print("Disparos:", disparos.shape)

## 4. Limpieza de los disparos: penaltis sin etiquetar

Understat deja el campo `situation` vacio en los penaltis. Se identifican porque todos
comparten el mismo xG (0,743278) y la misma posicion (`location_x` = 0,885, el punto
de penalti). Reconstruimos la categoria.

In [ ]:
# Convertimos el indice en columnas para no perder league, season y game al guardar
disparos = disparos.reset_index()

# Identificamos los penaltis por su posicion exacta en el punto de penalti
mascara_penalti = disparos["situation"].isna() & (disparos["location_x"] == 0.885)

# Etiquetamos esos disparos como penaltis
disparos.loc[mascara_penalti, "situation"] = "Penalty"

# Comprobamos que no queda ningun disparo sin clasificar
print("Disparos sin situacion:", disparos["situation"].isna().sum())

# Guardamos el dataset ya corregido
disparos.to_csv("data/disparos_laliga_2526_limpio.csv", index=False)

# Mostramos el reparto por tipo de jugada
print("\n", disparos["situation"].value_counts())

## 5. Reestructuracion de la tabla de equipos

Understat devuelve una fila por partido con las columnas duplicadas en `home_*` y `away_*`.
La convertimos a formato largo (una fila por equipo y partido) y calculamos los goles
y el xG **en contra**, necesarios para el analisis defensivo.

In [ ]:
# Reseteamos el indice para trabajar con las columnas libremente
df = equipos.reset_index()

# Definimos las metricas que existen tanto para local como para visitante
metricas = ["points", "expected_points", "goals", "xg", "np_xg",
            "np_xg_difference", "ppda", "deep_completions"]

# Construimos la tabla de los equipos locales
local = df[["league", "season", "game", "date", "home_team", "away_team"] +
           ["home_" + m for m in metricas]].copy()
# Renombramos las columnas quitando el prefijo home_
local.columns = ["league", "season", "game", "date", "equipo", "rival"] + metricas
# Marcamos que estos partidos se jugaron en casa
local["condicion"] = "casa"

# Construimos la tabla de los equipos visitantes
visitante = df[["league", "season", "game", "date", "away_team", "home_team"] +
               ["away_" + m for m in metricas]].copy()
# Renombramos las columnas quitando el prefijo away_
visitante.columns = ["league", "season", "game", "date", "equipo", "rival"] + metricas
# Marcamos que estos partidos se jugaron fuera
visitante["condicion"] = "fuera"

# Unimos ambas tablas en formato largo
largo = pd.concat([local, visitante], ignore_index=True)

# Preparamos los datos del rival para calcular lo encajado
contra = largo[["game", "equipo", "goals", "xg", "np_xg"]].rename(
    columns={"equipo": "rival", "goals": "goles_encajados",
             "xg": "xga", "np_xg": "np_xga"})

# Cruzamos para anadir goles y xG en contra de cada partido
largo = largo.merge(contra, on=["game", "rival"], how="left")

# Guardamos la tabla en formato largo
largo.to_csv("data/equipos_largo_laliga_2526.csv", index=False)

# Comprobamos que hay 760 filas (380 partidos x 2 equipos)
print("Formato largo:", largo.shape)

## 6. FBref: estadisticas de jugador de LaLiga

FBref usa Selenium: se abrira una ventana de Chrome que **no hay que cerrar**.
La libreria espera 7 segundos entre peticiones.

In [ ]:
# Creamos el lector de FBref para LaLiga temporada 2025-2026
fbref = sd.FBref(leagues="ESP-La Liga", seasons="2025-2026")

# Descargamos las estadisticas estandar de todos los jugadores
fb_standard = fbref.read_player_season_stats(stat_type="standard")
# Guardamos las estadisticas estandar
fb_standard.to_csv("data/fbref_standard_laliga_2526.csv")

# Descargamos las estadisticas miscelaneas (intercepciones, entradas, faltas, centros)
fb_misc = fbref.read_player_season_stats(stat_type="misc")
# Guardamos las estadisticas miscelaneas
fb_misc.to_csv("data/fbref_misc_laliga_2526.csv")

# Comprobamos las dimensiones
print("FBref standard:", fb_standard.shape)
print("FBref misc:", fb_misc.shape)

## 7. WhoScored: eventos Opta de LaLiga

Descarga el flujo completo de eventos de los 380 partidos: despejes, bloqueos, entradas,
duelos aereos y recuperaciones, todos con coordenadas.

El bucle guarda **un fichero por partido** y salta los ya descargados, asi que se puede
parar (boton de stop) y reanudar sin perder nada.

In [ ]:
# Creamos el lector de WhoScored para LaLiga temporada 2025-2026
ws = sd.WhoScored(leagues="ESP-La Liga", seasons="2025-2026")

# Leemos el calendario de WhoScored
cal_ws = ws.read_schedule().reset_index()

# Nos quedamos con los identificadores de todos los partidos
ids_partidos = cal_ws["game_id"].dropna().unique()

# Mostramos cuantos partidos hay que descargar
print("Partidos a descargar:", len(ids_partidos))

In [ ]:
# Recorremos todos los partidos uno a uno
for i, gid in enumerate(ids_partidos):
    # Construimos la ruta del fichero de ese partido
    ruta = f"data/eventos/eventos_{int(gid)}.parquet"
    # Si el partido ya esta descargado lo saltamos
    if os.path.exists(ruta):
        continue
    # Intentamos descargar y guardar los eventos del partido
    try:
        # Pedimos los eventos a WhoScored
        ev = ws.read_events(match_id=int(gid))
        # Guardamos los eventos de ese partido
        ev.to_parquet(ruta)
    # Si falla mostramos el error completo para poder diagnosticarlo
    except Exception as e:
        print(f"ERROR {int(gid)}: {type(e).__name__}: {e}")

# Contamos cuantos partidos han quedado guardados
print("Partidos guardados:", len(os.listdir("data/eventos")), "de", len(ids_partidos))

## 8. Unificacion de los eventos en un solo fichero

In [ ]:
# Leemos todos los ficheros de eventos y los unimos en una sola tabla
eventos = pd.concat([pd.read_parquet(f) for f in glob.glob("data/eventos/*.parquet")],
                    ignore_index=True)

# Guardamos el conjunto unificado para no repetir la lectura
eventos.to_parquet("data/eventos_laliga_2526.parquet")

# Comprobamos el volumen y que estan todos los partidos
print("Eventos totales:", len(eventos))
print("Partidos distintos:", eventos["game_id"].nunique())

## 9. Alineaciones: posiciones, edad y altura

soccerdata no expone estos campos, pero **si estan en el JSON en bruto** que guarda en
cache. Los extraemos directamente de ahi.

Cada ficha de partido incluye, por jugador: la posicion en la que jugo (`DC`, `DL`, `MC`...),
si fue titular, su edad y su altura. WhoScored solo registra posicion para los titulares:
los suplentes aparecen todos como `Sub`.

In [ ]:
# Construimos la ruta de la cache de eventos de LaLiga
carpeta_cache = os.path.expanduser("~/soccerdata/data/WhoScored/events/ESP-La Liga_2526")

# Preparamos la lista donde guardaremos cada jugador de cada partido
filas = []

# Recorremos todas las fichas de partido de la cache
for ruta in glob.glob(os.path.join(carpeta_cache, "*")):
    # Abrimos el fichero y lo leemos como JSON
    with open(ruta, encoding="utf-8") as f:
        d = json.load(f)
    # Saltamos los ficheros vacios
    if not d:
        continue
    # Recorremos el equipo local y el visitante
    for lado in ["home", "away"]:
        # Extraemos los datos del equipo
        equipo = d[lado]
        # Recorremos cada jugador convocado
        for p in equipo["players"]:
            # Guardamos los campos que nos interesan
            filas.append({
                "partido": os.path.basename(ruta),
                "team": equipo["name"],
                "player_id": p.get("playerId"),
                "player": p.get("name"),
                "posicion_ws": p.get("position"),
                "titular": p.get("isFirstEleven", False),
                "edad": p.get("age"),
                "altura": p.get("height")})

# Convertimos la lista en una tabla
alineaciones = pd.DataFrame(filas)

# Guardamos la tabla
alineaciones.to_csv("data/alineaciones_laliga_2526.csv", index=False)

# Mostramos el volumen obtenido
print("Filas:", len(alineaciones))
print("Jugadores distintos:", alineaciones["player_id"].nunique())

# Mostramos los codigos de posicion encontrados
print("\n", alineaciones["posicion_ws"].value_counts())

## 10. Configuracion de Segunda Division

soccerdata solo trae configuradas las cinco grandes ligas. Para anadir Segunda hay que
declararla en `league_dict.json`, con el nombre **exacto** que usa cada fuente.

El nombre de FBref (`Spanish Segunda División`) se obtuvo de su propio listado de
competiciones, no por suposicion.

**Tras ejecutar esta celda hay que reiniciar el kernel**: soccerdata lee la configuracion
al arrancar.

In [ ]:
# Construimos la ruta de la carpeta de configuracion de soccerdata
carpeta_config = os.path.expanduser("~/soccerdata/config")

# Creamos la carpeta si no existe
os.makedirs(carpeta_config, exist_ok=True)

# Definimos la configuracion de Segunda Division
# WhoScored se omite a proposito: no publica eventos para esta categoria
config_ligas = {
    "ESP-Segunda Division": {
        "FBref": "Spanish Segunda División",
        "season_start": "Aug",
        "season_end": "Jun"}}

# Escribimos el fichero de configuracion
with open(os.path.join(carpeta_config, "league_dict.json"), "w", encoding="utf-8") as f:
    json.dump(config_ligas, f, ensure_ascii=False, indent=2)

# Confirmamos la escritura
print("Configuracion guardada. Reiniciar el kernel antes de continuar.")

## 11. FBref: estadisticas de jugador de Segunda Division

In [ ]:
# Creamos el lector de FBref para Segunda Division
fbref2 = sd.FBref(leagues="ESP-Segunda Division", seasons="2025-2026")

# Descargamos las estadisticas miscelaneas de los jugadores
fb2_misc = fbref2.read_player_season_stats(stat_type="misc")

# Guardamos los datos descargados
fb2_misc.to_csv("data/fbref_misc_segunda_2526.csv")

# Comprobamos las dimensiones
print("Jugadores en Segunda:", fb2_misc.shape)

## 12. Exportacion para Power BI y Tableau

Tablas ya preparadas y relacionables entre si por `clave_equipo` y `clave`.
Se usa `utf-8-sig` para que los acentos se lean correctamente en Power BI y Excel.

In [ ]:
# Copiamos las tablas ya procesadas a la carpeta de exportacion
for origen, destino in [
    ("data/equipos_largo_laliga_2526.csv", "equipos_partido.csv"),
    ("data/disparos_laliga_2526_limpio.csv", "disparos.csv"),
    ("data/alineaciones_laliga_2526.csv", "alineaciones.csv")]:
    # Comprobamos que el fichero de origen existe
    if os.path.exists(origen):
        # Lo leemos y lo volvemos a guardar con la codificacion adecuada
        pd.read_csv(origen).to_csv(f"data/export_bi/{destino}", index=False, encoding="utf-8-sig")

# Copiamos tambien el fichero de eventos, que Power BI lee en formato Parquet
import shutil
shutil.copy("data/eventos_laliga_2526.parquet", "data/export_bi/eventos.parquet")

# Mostramos el contenido de la carpeta de exportacion
for f in sorted(os.listdir("data/export_bi")):
    ruta = os.path.join("data/export_bi", f)
    print(f"{f:28} {round(os.path.getsize(ruta)/1024/1024, 1):>6} MB")

---
### Ficheros generados

| Fichero | Contenido |
|---|---|
| `calendario_laliga_2526.csv` | Calendario de los 380 partidos |
| `equipos_partido_laliga_2526.csv` | Estadisticas de equipo por partido (formato original) |
| `equipos_largo_laliga_2526.csv` | Lo anterior en formato largo, con xG en contra |
| `jugadores_laliga_2526.csv` | Estadisticas de jugador de Understat (xG, xA, xGBuildup) |
| `disparos_laliga_2526_limpio.csv` | 9.523 disparos con coordenadas, xG y penaltis etiquetados |
| `fbref_misc_laliga_2526.csv` | Intercepciones, entradas, faltas y centros (Primera) |
| `fbref_misc_segunda_2526.csv` | Lo mismo para Segunda Division (698 jugadores) |
| `eventos_laliga_2526.parquet` | 577.205 eventos Opta de los 380 partidos |
| `alineaciones_laliga_2526.csv` | Posicion, titularidad, edad y altura por partido |

El fichero `posiciones_laliga_2526.csv` **no se descarga aqui**: es de elaboracion propia
a partir de las fichas de plantilla de Marca. Se documenta en el notebook 03.